# Olist Exploratory Data Analysis (EDA)
## Author: Ahmed Mohamed Awadalla
## Date: May 2026

This notebook performs exploratory data analysis on the cleaned Olist dataset to uncover insights about sales, customers, delivery, and reviews.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("Libraries loaded successfully")

## 2. Load Cleaned Data

In [ ]:
# Set path to cleaned data
clean_data_path = "../data/cleaned/"

# Load all cleaned tables
orders = pd.read_csv(clean_data_path + "orders_clean.csv")
customers = pd.read_csv(clean_data_path + "customers_clean.csv")
products = pd.read_csv(clean_data_path + "products_clean.csv")
sellers = pd.read_csv(clean_data_path + "sellers_clean.csv")
order_items = pd.read_csv(clean_data_path + "order_items_clean.csv")
payments = pd.read_csv(clean_data_path + "order_payments_clean.csv")
reviews = pd.read_csv(clean_data_path + "order_reviews_clean.csv")
geolocation = pd.read_csv(clean_data_path + "geolocation_clean.csv")

# Convert date columns back to datetime
date_cols = ['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col])

print("All cleaned data loaded successfully")
print(f"Orders: {orders.shape}")
print(f"Order Items: {order_items.shape}")
print(f"Payments: {payments.shape}")
print(f"Reviews: {reviews.shape}")

## 3. Sales & Revenue Analysis

### 3.1 Total Revenue and Key Metrics

In [ ]:
# Calculate revenue per order item
order_items['total_value'] = order_items['price'] + order_items['freight_value']

# Merge with orders to filter only delivered
delivered_orders = orders[orders['order_status'] == 'delivered']
delivered_items = order_items[order_items['order_id'].isin(delivered_orders['order_id'])]

# Calculate key metrics
total_revenue = delivered_items['total_value'].sum()
total_orders = delivered_orders['order_id'].nunique()
aov = total_revenue / total_orders

print("=" * 50)
print("SALES & REVENUE KPIs")
print("=" * 50)
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Delivered Orders: {total_orders:,}")
print(f"Average Order Value (AOV): ${aov:.2f}")

### 3.2 Monthly Revenue Trend

In [ ]:
# Create monthly revenue aggregation
orders_with_revenue = delivered_orders.merge(
    delivered_items.groupby('order_id')['total_value'].sum().reset_index(),
    on='order_id'
)

orders_with_revenue['year_month'] = orders_with_revenue['order_purchase_timestamp'].dt.to_period('M')
monthly_revenue = orders_with_revenue.groupby('year_month')['total_value'].sum().reset_index()
monthly_revenue['year_month_str'] = monthly_revenue['year_month'].astype(str)

# Plot monthly revenue trend
plt.figure(figsize=(14, 6))
plt.plot(monthly_revenue['year_month_str'], monthly_revenue['total_value'], 
         marker='o', linewidth=2, markersize=6, color='#2E86AB')
plt.title('Monthly Revenue Trend (Oct 2016 - Aug 2018)', fontsize=16, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Revenue ($)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nTop 5 Months by Revenue:")
print(monthly_revenue.sort_values('total_value', ascending=False).head())

### 3.3 Top Product Categories by Revenue

In [ ]:
# Merge with products to get category names
items_with_cat = delivered_items.merge(products, on='product_id', how='left')

# Aggregate by category
category_revenue = items_with_cat.groupby('product_category_name_english')['total_value'].sum().sort_values(ascending=False)

# Plot top 10 categories
plt.figure(figsize=(12, 8))
top_categories = category_revenue.head(10)
colors = plt.cm.Blues(np.linspace(0.4, 0.9, 10))
plt.barh(range(len(top_categories)), top_categories.values, color=colors)
plt.yticks(range(len(top_categories)), top_categories.index)
plt.xlabel('Revenue ($)', fontsize=12)
plt.title('Top 10 Product Categories by Revenue', fontsize=16, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 Categories by Revenue:")
for i, (cat, rev) in enumerate(top_categories.items(), 1):
    print(f"{i}. {cat}: ${rev:,.2f}")

## 4. Geographic Analysis

### 4.1 Revenue by State

In [ ]:
# Merge orders with customers to get state
orders_with_customer = delivered_orders.merge(customers, on='customer_id', how='left')
orders_with_customer_rev = orders_with_customer.merge(
    delivered_items.groupby('order_id')['total_value'].sum().reset_index(),
    on='order_id'
)

# Revenue by state
state_revenue = orders_with_customer_rev.groupby('customer_state')['total_value'].sum().sort_values(ascending=False)

# Plot top 10 states
plt.figure(figsize=(12, 6))
top_states = state_revenue.head(10)
sns.barplot(x=top_states.values, y=top_states.index, palette='Greens_r')
plt.xlabel('Revenue ($)', fontsize=12)
plt.title('Top 10 States by Revenue', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 5 States by Revenue:")
for i, (state, rev) in enumerate(state_revenue.head().items(), 1):
    pct = (rev / total_revenue) * 100
    print(f"{i}. {state}: ${rev:,.2f} ({pct:.1f}% of total)")

## 5. Delivery Performance Analysis

### 5.1 Delivery Time Distribution

In [ ]:
# Filter delivered orders with valid delivery diff
delivered_with_delay = delivered_orders[delivered_orders['delivery_diff_days'].notna()]

# Calculate on-time vs delayed
delivered_with_delay['is_on_time'] = delivered_with_delay['delivery_diff_days'] >= 0
on_time_count = delivered_with_delay['is_on_time'].sum()
delayed_count = (~delivered_with_delay['is_on_time']).sum()
on_time_rate = (on_time_count / len(delivered_with_delay)) * 100

print("=" * 50)
print("DELIVERY PERFORMANCE")
print("=" * 50)
print(f"On-Time Deliveries: {on_time_count:,}")
print(f"Delayed Deliveries: {delayed_count:,}")
print(f"On-Time Delivery Rate: {on_time_rate:.2f}%")
print(f"Average Delivery Days: {delivered_with_delay['actual_delivery_days'].mean():.2f} days")
print(f"Average Delay (for delayed orders): {delivered_with_delay[delivered_with_delay['delivery_diff_days'] < 0]['delivery_diff_days'].mean():.2f} days")

In [ ]:
# Plot delivery time distribution
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
delivered_with_delay['actual_delivery_days'].hist(bins=30, color='#5D9B9B', edgecolor='black')
plt.xlabel('Actual Delivery Days')
plt.ylabel('Number of Orders')
plt.title('Distribution of Actual Delivery Days')

plt.subplot(1, 2, 2)
delivered_with_delay['delivery_diff_days'].hist(bins=30, color='#A37C6E', edgecolor='black')
plt.axvline(x=0, color='red', linestyle='--', linewidth=2, label='On-Time Threshold')
plt.xlabel('Delivery Difference (Actual - Estimated)')
plt.ylabel('Number of Orders')
plt.title('Distribution of Delivery Difference')
plt.legend()

plt.tight_layout()
plt.show()

### 5.2 Delivery Time vs Review Score

In [ ]:
# Merge reviews with delivery data
reviews_with_delivery = reviews.merge(
    delivered_with_delay[['order_id', 'delivery_diff_days', 'actual_delivery_days']],
    on='order_id',
    how='inner'
)

# Calculate average delivery days by review score
delivery_by_score = reviews_with_delivery.groupby('review_score')['delivery_diff_days'].mean().reset_index()

# Plot
plt.figure(figsize=(10, 6))
colors = ['#D9534F', '#F0AD4E', '#5BC0DE', '#5CB85C', '#4CAF50']
plt.bar(delivery_by_score['review_score'], delivery_by_score['delivery_diff_days'], color=colors)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2, label='On-Time Threshold')
plt.xlabel('Review Score', fontsize=12)
plt.ylabel('Average Delivery Difference (Days)', fontsize=12)
plt.title('Impact of Delivery Time on Customer Reviews', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print("\nAverage delivery difference by review score:")
for _, row in delivery_by_score.iterrows():
    print(f"  Score {int(row['review_score'])}: {row['delivery_diff_days']:.2f} days")

## 6. Payment Analysis

### 6.1 Payment Method Distribution

In [ ]:
# Payment method usage
payment_counts = payments['payment_type'].value_counts()
payment_pct = (payment_counts / len(payments)) * 100

# Plot
plt.figure(figsize=(10, 6))
plt.pie(payment_counts.values, labels=payment_counts.index, autopct='%1.1f%%', 
        startangle=90, colors=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D'])
plt.title('Payment Method Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nPayment Method Breakdown:")
for method, pct in payment_pct.items():
    print(f"  {method}: {pct:.1f}%")

### 6.2 Average Installments by Payment Method

In [ ]:
installments_by_method = payments.groupby('payment_type')['payment_installments'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=installments_by_method.values, y=installments_by_method.index, palette='Purples_r')
plt.xlabel('Average Number of Installments', fontsize=12)
plt.title('Average Installments by Payment Method', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nAverage Installments by Payment Method:")
for method, inst in installments_by_method.items():
    print(f"  {method}: {inst:.1f} installments")

## 7. Customer Analysis

### 7.1 Repeat Customer Rate

In [ ]:
# Count orders per customer
orders_per_customer = delivered_orders.merge(customers, on='customer_id').groupby('customer_unique_id')['order_id'].nunique()

repeat_customers = (orders_per_customer > 1).sum()
total_customers = len(orders_per_customer)
repeat_rate = (repeat_customers / total_customers) * 100

print("=" * 50)
print("CUSTOMER ANALYSIS")
print("=" * 50)
print(f"Total Unique Customers: {total_customers:,}")
print(f"Repeat Customers: {repeat_customers:,}")
print(f"Repeat Customer Rate: {repeat_rate:.2f}%")
print(f"One-Time Buyers: {total_customers - repeat_customers:,}")

# Plot customer distribution
customer_dist = orders_per_customer.value_counts().sort_index()

plt.figure(figsize=(12, 6))
plt.bar(customer_dist.index[:10], customer_dist.values[:10], color='#5D9B9B')
plt.xlabel('Number of Orders per Customer', fontsize=12)
plt.ylabel('Number of Customers', fontsize=12)
plt.title('Customer Order Frequency Distribution (Top 10)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.2 Review Score Distribution

In [ ]:
score_distribution = reviews['review_score'].value_counts().sort_index()

plt.figure(figsize=(10, 6))
colors = ['#D9534F', '#F0AD4E', '#5BC0DE', '#5CB85C', '#4CAF50']
plt.bar(score_distribution.index, score_distribution.values, color=colors, edgecolor='black')
plt.xlabel('Review Score', fontsize=12)
plt.ylabel('Number of Reviews', fontsize=12)
plt.title('Customer Review Score Distribution', fontsize=14, fontweight='bold')
plt.xticks(range(1, 6))

# Add percentage labels
total_reviews = len(reviews)
for i, (score, count) in enumerate(score_distribution.items()):
    pct = (count / total_reviews) * 100
    plt.text(score, count + 500, f'{pct:.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

avg_score = reviews['review_score'].mean()
print(f"\nAverage Review Score: {avg_score:.2f} / 5.0")

## 8. Correlation Analysis

In [ ]:
# Create correlation dataset
correlation_data = delivered_with_delay[['actual_delivery_days', 'delivery_diff_days']].copy()
correlation_data['price'] = delivered_items.groupby('order_id')['price'].sum().values
correlation_data['freight'] = delivered_items.groupby('order_id')['freight_value'].sum().values
correlation_data['total_value'] = delivered_items.groupby('order_id')['total_value'].sum().values

# Add review scores
review_avg = reviews.groupby('order_id')['review_score'].mean()
correlation_data['review_score'] = correlation_data.index.map(review_avg)

# Calculate correlation matrix
corr_matrix = correlation_data.corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, fmt='.2f')
plt.title('Correlation Matrix of Key Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey Correlations:")
print(f"  Delivery delay vs Review Score: {corr_matrix.loc['delivery_diff_days', 'review_score']:.2f}")
print(f"  Price vs Review Score: {corr_matrix.loc['price', 'review_score']:.2f}")
print(f"  Freight vs Total Value: {corr_matrix.loc['freight', 'total_value']:.2f}")

## 9. Key Insights Summary

In [ ]:
print("=" * 60)
print("KEY INSIGHTS SUMMARY")
print("=" * 60)

print("\n📊 SALES & REVENUE:")
print(f"  • Total Revenue: ${total_revenue:,.2f}")
print(f"  • Average Order Value: ${aov:.2f}")
print(f"  • Best Category: {top_categories.index[0]} (${top_categories.values[0]:,.2f})")

print("\n📍 GEOGRAPHIC:")
print(f"  • Best State: {state_revenue.index[0]} ({state_revenue.values[0]/total_revenue*100:.1f}% of revenue)")

print("\n🚚 DELIVERY:")
print(f"  • On-Time Delivery Rate: {on_time_rate:.2f}%")
print(f"  • Average Delivery Days: {delivered_with_delay['actual_delivery_days'].mean():.2f}")

print("\n⭐ CUSTOMER SATISFACTION:")
print(f"  • Average Review Score: {avg_score:.2f}/5")
print(f"  • Repeat Customer Rate: {repeat_rate:.2f}%")

print("\n💳 PAYMENTS:")
print(f"  • Most Used Payment: {payment_counts.index[0]} ({payment_pct.iloc[0]:.1f}%)")

print("\n" + "=" * 60)
print("EDA COMPLETED SUCCESSFULLY")
print("=" * 60)